1. Ver las tablas del db.

In [1]:
import pandas as pd
import sqlite3

In [2]:
def sql_query(cursor, query):
    cursor.execute(query)
    ans = cursor.fetchall()
    names = [description[0] for description in cursor.description]
    return pd.DataFrame(ans,columns=names)

In [3]:
connection = sqlite3.connect("./db/sql-murder-mystery.db")
cursor_misterio = connection.cursor()

In [4]:
query = '''
SELECT name 
  FROM sqlite_master
 where type = 'table'
'''
sql_query(cursor_misterio, query)

,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


2. Ver la estructura de *crime_scene_report*.

In [5]:
query = '''
SELECT sql 
  FROM sqlite_master
 where name = 'crime_scene_report'
'''
cursor_misterio.execute(query)
report = cursor_misterio.fetchall()
report

[('CREATE TABLE crime_scene_report (\n        date integer,\n        type text,\n        description text,\n        city text\n    )',)]

3. Vamos a buscar el caso partiendo de la pista inicial: "_You vaguely remember that the crime was a ​murder​ that occurred sometime on ​Jan.15, 2018​_"

In [6]:
query = """
SELECT *
FROM crime_scene_report
WHERE date == 20180115 AND city == "SQL City" AND type == "murder"
"""
caso = sql_query(cursor_misterio, query)
caso

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [7]:
# Veamos la información del caso
print(caso.description[0])

Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".


> Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".

In [8]:
# Buscamos a los testigos, el primero vive en Northwestern Dr
query = """
SELECT *
FROM person
WHERE address_street_name == "Northwestern Dr"
"""
testigos_1 = sql_query(cursor_misterio, query)
testigos_1.sort_values(by = ["address_number"], ascending = False).head()  # vive en la última casa

,id,name,license_id,address_number,address_street_name,ssn
2,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
4,17729,Lasonya Wildey,439686,3824,Northwestern Dr,917817122
27,53890,Sophie Tiberio,957671,3755,Northwestern Dr,442830147
35,73368,Torie Thalmann,773862,3697,Northwestern Dr,341559436
47,96595,Coretta Cubie,303645,3631,Northwestern Dr,378403829


Vamos a ver la entrevista de Morty Schapiro

In [9]:
query = """
SELECT transcript
FROM interview
WHERE person_id == 	14887
"""
print(sql_query(cursor_misterio, query)["transcript"][0])

I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".


El primer sospechoso es un hombre, miembro de oro en el gimnasio, con número de socio empezando en "48Z" (get_fit_now_member) y matrícula que incluye "H42W" (drivers_license)

> I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W"

In [10]:
query = """
SELECT p.*, dl.gender, dl.plate_number, gf.membership_status, gf.id
FROM get_fit_now_member as gf
LEFT JOIN person as p ON gf.person_id == p.id
LEFT JOIN drivers_license as dl ON p.license_id == dl.id 
WHERE gf.membership_status == "gold" AND gf.id LIKE "48Z%" AND dl.gender == "male" AND dl.plate_number LIKE "%H42W%"
"""
sql_query(cursor_misterio, query)

,id,name,license_id,address_number,address_street_name,ssn,gender,plate_number,membership_status,id
0,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279,male,0H42W2,gold,48Z55


In [11]:
# Veamos qué tiene que decir
query = """
SELECT transcript
FROM interview
WHERE person_id == 	67318
"""
print(sql_query(cursor_misterio, query)["transcript"][0])

I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.



Este debería ir a la cárcel, pero vamos a seguir buscando a la otra.
> I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.

- Altura entre 65 y 67
- Pelo rojo
- Conduce un Tesla Model S.
- Fue a ver SQL Symphony Concert 3 veces en Diciembre de 2017

In [ ]:
query = """
SELECT p.id as id_personal, p.name, p.ssn, i.annual_income, dl.*
FROM drivers_license as dl
LEFT JOIN person as p ON p.license_id == dl.id
LEFT JOIN income as i ON p.ssn = i.ssn
WHERE (dl.height > 65 AND dl.height < 67) AND dl.gender == "female" AND dl.hair_color = "red" \
    AND dl.car_make == "Tesla" AND dl.car_model == "Model S"
"""
# veamos quienes son las sospechosas principales
sql_query(cursor_misterio, query)

,id_personal,name,ssn,annual_income,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,99716,Miranda Priestly,987756388,310000.0,202298,68,66,green,red,female,500123,Tesla,Model S
1,90700,Regina George,337169072,NaN,291182,65,66,blue,red,female,08CM64,Tesla,Model S


In [ ]:
# También sabemos que fue al teatro 3 veces en diciembre de 2017 para ver SQL Symphony Concert
query = """
SELECT f.event_name, p.name, COUNT(p.name) as numero_eventos
FROM facebook_event_checkin as f
LEFT JOIN person as p ON f.person_id == p.id
WHERE f.date LIKE "201712%" AND f.event_name == "SQL Symphony Concert" AND p.id IN (99716,90700)
GROUP BY p.name
ORDER BY 3 DESC
"""
sql_query(cursor_misterio, query).head()

,event_name,name,numero_eventos
0,SQL Symphony Concert,Miranda Priestly,3


Comprobando en la página web, hemos resuelto el caso y Miranda Priestly es la culpable. Me sobra la información del segundo testigo...

In [13]:
# El otro testigo se llama Annabel y vive en Franklin Ave
query = """
SELECT *
FROM person
WHERE name LIKE "Annabel%" AND address_street_name == "Franklin Ave"
"""
testigos_2 = sql_query(cursor_misterio, query)
testigos_2

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


In [22]:
# Vamos a ahondar en su entrevista
query = """
SELECT transcript
FROM interview
WHERE person_id == 16371
"""
print(sql_query(cursor_misterio, query)["transcript"][0])

I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


Seguimos las pistas hasta los checkins del gimnasio

In [ ]:
query = """
SELECT a.*, b.*
FROM get_fit_now_check_in as a
LEFT JOIN get_fit_now_member as b
WHERE a.membership_id == b.id AND a.check_in_date NOT LIKE "2017%"  -- la semana pasada al asesinato no puede ser 2017 
"""
gimnasio_2 = sql_query(cursor_misterio, query)
gimnasio_2

,membership_id,check_in_date,check_in_time,check_out_time,id,person_id,name,membership_start_date,membership_status
0,NL318,20180212,329,365,NL318,65076,Everette Koepke,20170926,gold
1,NL318,20180429,506,554,NL318,65076,Everette Koepke,20170926,gold
2,NL318,20180128,124,759,NL318,65076,Everette Koepke,20170926,gold
3,NL318,20180114,777,813,NL318,65076,Everette Koepke,20170926,gold
4,NL318,20180110,476,498,NL318,65076,Everette Koepke,20170926,gold
...,...,...,...,...,...,...,...,...,...
667,4KB72,20180303,313,830,4KB72,79110,Emile Hege,20170522,regular
668,4KB72,20180126,318,680,4KB72,79110,Emile Hege,20170522,regular
669,48Z7A,20180109,1600,1730,48Z7A,28819,Joe Germuska,20160305,gold
670,48Z55,20180109,1530,1700,48Z55,67318,Jeremy Bowers,20160101,gold


In [ ]:
sospechosos_gimnasio_2 = gimnasio_2.loc[gimnasio.check_in_date.astype("str").str.contains("0109"), ]   # todos los valores quí tienen que ser únicos
sospechosos_gimnasio_2

,membership_id,check_in_date,check_in_time,check_out_time,id,person_id,name,membership_start_date,membership_status
31,X0643,20180109,957,1164,X0643,15247,Shondra Ledlow,20170521,silver
217,UK1F2,20180109,344,518,UK1F2,28073,Zackary Cabotage,20170818,silver
322,XTE42,20180109,486,1124,XTE42,55662,Sarita Bartosh,20170524,gold
353,1AE2H,20180109,461,944,1AE2H,10815,Adriane Pelligra,20170816,silver
428,6LSTG,20180109,399,515,6LSTG,83186,Burton Grippe,20170214,gold
529,7MWHJ,20180109,273,885,7MWHJ,31523,Blossom Crescenzo,20180309,regular
604,GE5Q8,20180109,367,959,GE5Q8,92736,Carmen Dimick,20170618,gold
669,48Z7A,20180109,1600,1730,48Z7A,28819,Joe Germuska,20160305,gold
670,48Z55,20180109,1530,1700,48Z55,67318,Jeremy Bowers,20160101,gold
671,90081,20180109,1600,1700,90081,16371,Annabel Miller,20160208,gold


In [ ]:
# El checkout del sospechoso debe ser más tarde del checkin de Annabel si se han cruzado, 
# y el checkin antes del checkout de Annabel
sospechosos_gimnasio_2.drop(sospechosos_gimnasio_2.loc[(sospechosos_gimnasio_2.check_out_time < sospechosos_gimnasio_2.loc[671, "check_in_time"]) | \
    (sospechosos_gimnasio_2.check_in_time > sospechosos_gimnasio_2.loc[671, "check_out_time"])].index, axis = 0)   # como es un drop, las condiciones van al revés (y es un OR)

,membership_id,check_in_date,check_in_time,check_out_time,id,person_id,name,membership_start_date,membership_status
669,48Z7A,20180109,1600,1730,48Z7A,28819,Joe Germuska,20160305,gold
670,48Z55,20180109,1530,1700,48Z55,67318,Jeremy Bowers,20160101,gold
671,90081,20180109,1600,1700,90081,16371,Annabel Miller,20160208,gold
